In [22]:
# !pip install pandas numpy tqdm requests rapidfuzz python-dotenv

### Анализ англоязычного корпуса

In [7]:
import os
import re
import json
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from rapidfuzz import fuzz
import string
from deep_translator import GoogleTranslator
from sentence_transformers import SentenceTransformer, util
import pymorphy2
from pathlib import Path
from collections import Counter, defaultdict

MAIN_DIR = (Path.cwd()).parent

all_labels = []
EN_CORPUS = MAIN_DIR / 'en_corpus/datasets/train-labels-FLC'

for f in os.listdir(EN_CORPUS):
    if f.startswith('.') or 'Zone.Identifier' in f:
        continue
    
    if not f.endswith('.labels') or 'FLC' not in f:
        continue
    
    file_path = os.path.join(EN_CORPUS, f)
    
    try:
        df = pd.read_csv(file_path, sep='\t', header=None)
        
        if df.shape[1] >= 2:
            all_labels.extend(df[1].tolist())
        else:
            print(f"Предупреждение: в файле {f} меньше 2 колонок")
            
    except Exception as e:
        print(f"Ошибка при чтении файла {f}: {e}")

print(f"Всего обработано спанов: {len(all_labels)}")
print("\nРаспределение техник:")
counter = Counter(all_labels)
for technique, count in counter.most_common():
    print(f"{technique}: {count}")

Ошибка при чтении файла article754348478.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article730652769.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article758882558.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article758469195.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article695108099.task-FLC.labels: No columns to parse from file


Ошибка при чтении файла article7385423989.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article728153988.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article729700539.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article758385628.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article725238842.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article758669180.task-FLC.labels: No columns to parse from file
Ошибка при чтении файла article754508491.task-FLC.labels: No columns to parse from file
Всего обработано спанов: 6041

Распределение техник:
Loaded_Language: 2115
Name_Calling,Labeling: 1085
Repetition: 571
Doubt: 490
Exaggeration,Minimisation: 479
Flag-Waving: 240
Appeal_to_fear-prejudice: 239
Causal_Oversimplification: 201
Slogans: 136
Appeal_to_Authority: 116
Black-and-White_Fallacy: 109
Thought-terminating_Cliches: 79
Whataboutism: 57
Reductio_ad_hitlerum: 54
Red

### Парсинг и унификация англоязычного корпуса

In [8]:
# Сопоставление: исходная техника - унифицированная метка (Таблица 3)
TECHNIQUE_TO_UNIFIED = {
    # fear_uncertainty_pressure
    'Appeal_to_fear-prejudice': 'fear_uncertainty_pressure',
    'Appeal_to_fear-prejudice,': 'fear_uncertainty_pressure',
    'Appeal_to_Fear-Prejudice': 'fear_uncertainty_pressure',
    'Appeal to fear/prejudice': 'fear_uncertainty_pressure',
    
    # emotional_triggering_language
    'Loaded_Language': 'emotional_triggering_language',
    
    # doubt_uncertainty_injection
    'Doubt': 'doubt_uncertainty_injection',
    
    # cognitive_closure_cliches
    'Thought-terminating_Cliches': 'cognitive_closure_cliches',
    'Thought-terminating cliches': 'cognitive_closure_cliches',
    'Thought-terminating_Cliches,': 'cognitive_closure_cliches',
    
    # social_proof_pressure
    'Bandwagon': 'social_proof_pressure',
    
    # gain_loss_exaggeration
    'Exaggeration,Minimisation': 'gain_loss_exaggeration',
    'Exaggeration,Minimisation,': 'gain_loss_exaggeration',
    'Exaggeration, Minimisation': 'gain_loss_exaggeration',
    'Exaggeration/Minimisation': 'gain_loss_exaggeration',
    
    # causal_fact_distortion
    'Causal_Oversimplification': 'causal_fact_distortion',
    'Causal Oversimplification': 'causal_fact_distortion',
    'Black-and-White_Fallacy': 'causal_fact_distortion',
    'Black-and-White_Fallacy,': 'causal_fact_distortion',
    'Black-and-White Fallacy': 'causal_fact_distortion',
    
    # authority_claim_pressure
    'Appeal_to_Authority': 'authority_claim_pressure',
    'Appeal to Authority': 'authority_claim_pressure',
    
    # topic_shift_misrepresentation
    'Whataboutism': 'topic_shift_misrepresentation',
    'Whataboutism,': 'topic_shift_misrepresentation',
    'Straw_Men': 'topic_shift_misrepresentation',
    'Straw Men': 'topic_shift_misrepresentation',
    'Straw_Men,': 'topic_shift_misrepresentation',
    'Red_Herring': 'topic_shift_misrepresentation',
    'Red Herring': 'topic_shift_misrepresentation',
    'Red_Herring,': 'topic_shift_misrepresentation',
    
    # directive_action_pressure
    'Repetition': 'directive_action_pressure',
}


EXCLUDED_TECHNIQUES = {
    'Name_Calling,Labeling', 'Name_Calling', 'Labeling',
    'Flag-Waving', 'Flag_Waving', 'Flag Waving',
    'Slogans',
    'Reductio_ad_hitlerum', 'Reductio ad hitlerum',
    'Obfuscation,Intentional_Vagueness,Confusion',
}


def normalize_technique_name(name):
    name = name.strip().replace(' ', '_').replace(',', '_').replace('/', '_')

    while '__' in name:
        name = name.replace('__', '_')
    return name


def parse_labels_file(file_path):
    spans = []
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            for line_num, line in enumerate(f, 1):
                line = line.strip()
                if not line:
                    continue
                
                parts = line.split('\t')
                

                if len(parts) < 4:
                    
                    if len(parts) == 3:
                        technique, start, end = parts
                        span_id = f"span_{line_num}"
                    else:
                        continue
                else:

                    span_id = parts[-4] if len(parts) >= 4 else f"span_{line_num}"
                    technique = parts[-3] if len(parts) >= 4 else parts[0]
                    start = parts[-2] if len(parts) >= 4 else parts[1]
                    end = parts[-1] if len(parts) >= 4 else parts[2]
                

                technique_clean = normalize_technique_name(technique)
                

                if technique in EXCLUDED_TECHNIQUES or technique_clean in [normalize_technique_name(t) for t in EXCLUDED_TECHNIQUES]:
                    continue
                

                unified_label = None
                for key, value in TECHNIQUE_TO_UNIFIED.items():
                    if normalize_technique_name(key) == technique_clean:
                        unified_label = value
                        break
                
                if unified_label is None:
  
                    for key, value in TECHNIQUE_TO_UNIFIED.items():
                        if technique_clean in normalize_technique_name(key) or normalize_technique_name(key) in technique_clean:
                            unified_label = value
                            print(f"Нечеткое сопоставление: {technique} → {unified_label}")
                            break
                
                if unified_label is None:

                    is_excluded = False
                    for exc in EXCLUDED_TECHNIQUES:
                        if technique_clean in normalize_technique_name(exc):
                            is_excluded = True
                            break
                    
                    if is_excluded:
                        continue
                    else:
                        print(f"ВНИМАНИЕ: Неизвестная техника: '{technique}' (cleaned: '{technique_clean}') в файле {file_path}")
                        continue
                
                spans.append({
                    'span_id': span_id,
                    'original_technique': technique,
                    'unified_label': unified_label,
                    'start': int(start),
                    'end': int(end),
                    'file': os.path.basename(file_path)
                })
    
    except Exception as e:
        print(f"Ошибка при парсинге {file_path}: {e}")
    
    return spans


def load_article_text(article_id, articles_dir):

    file_path = os.path.join(articles_dir, f"{article_id}.txt")
    if not os.path.exists(file_path):

        file_path = os.path.join(articles_dir, f"{article_id.replace('article', '')}.txt")
    
    if os.path.exists(file_path):
        with open(file_path, 'r', encoding='utf-8') as f:
            return f.read()
    return None


def process_dataset(labels_dir, articles_dir=None):
    all_spans = []
    file_stats = defaultdict(lambda: defaultdict(int))
    empty_files = []
    error_files = []
    
    for filename in os.listdir(labels_dir):
        if filename.startswith('.') or 'Zone.Identifier' in filename:
            continue
        
        if not (filename.endswith('.labels') and 'FLC' in filename):
            continue
        
        file_path = os.path.join(labels_dir, filename)
        
        spans = parse_labels_file(file_path)
        
        if not spans:
            try:
                with open(file_path, 'r', encoding='utf-8') as f:
                    content = f.read().strip()
                    if not content:
                        empty_files.append(filename)
                    else:
                        error_files.append(filename)
            except:
                error_files.append(filename)
            continue
        
        all_spans.extend(spans)
        

        for span in spans:
            file_stats[filename][span['unified_label']] += 1
        

        if articles_dir:
            article_id = filename.replace('.task-FLC.labels', '')
            article_text = load_article_text(article_id, articles_dir)
            if article_text:
                for span in spans:
                    span['text'] = article_text[span['start']:span['end']]
    

    global_stats = Counter()
    for span in all_spans:
        global_stats[span['unified_label']] += 1
    
    return {
        'spans': all_spans,
        'global_stats': global_stats,
        'file_stats': dict(file_stats),
        'empty_files': empty_files,
        'error_files': error_files,
        'total_spans': len(all_spans),
        'total_files': len(file_stats) + len(empty_files)
    }


LABELS_DIR = MAIN_DIR / 'en_corpus/datasets/train-labels-FLC'
ARTICLES_DIR = MAIN_DIR / 'en_corpus/datasets/train-articles'


results = process_dataset(LABELS_DIR, ARTICLES_DIR if os.path.exists(ARTICLES_DIR) else None)

print(f"Всего файлов разметки: {results['total_files']}")
print(f"Пустых файлов: {len(results['empty_files'])}")
print(f"Файлов с ошибками: {len(results['error_files'])}")
print(f"Всего спанов после фильтрации: {results['total_spans']}")

print(f"\nРаспределение унифицированных меток:")
for label, count in results['global_stats'].most_common():
    percentage = (count / results['total_spans']) * 100
    print(f"   {label}: {count} ({percentage:.1f}%)")

print(f"\nПримеры 5 файлов с разметкой леблов:")
for i, (filename, stats) in enumerate(list(results['file_stats'].items())[:5]):
    print(f"   {filename}: {dict(stats)}")


print(f"\nПримеры спанов:")
for span in results['spans'][:10]:
    print(f"   [{span['unified_label']}] {span.get('text', '(текст не загружен)')[:80]}...")

Всего файлов разметки: 349
Пустых файлов: 12
Файлов с ошибками: 1
Всего спанов после фильтрации: 4515

Распределение унифицированных меток:
   emotional_triggering_language: 2115 (46.8%)
   directive_action_pressure: 571 (12.6%)
   doubt_uncertainty_injection: 490 (10.9%)
   gain_loss_exaggeration: 479 (10.6%)
   causal_fact_distortion: 310 (6.9%)
   fear_uncertainty_pressure: 239 (5.3%)
   authority_claim_pressure: 116 (2.6%)
   topic_shift_misrepresentation: 103 (2.3%)
   cognitive_closure_cliches: 79 (1.7%)
   social_proof_pressure: 13 (0.3%)

Примеры 5 файлов с разметкой леблов:
   article770156851.task-FLC.labels: {'emotional_triggering_language': 14, 'causal_fact_distortion': 3, 'directive_action_pressure': 1, 'doubt_uncertainty_injection': 1, 'cognitive_closure_cliches': 2}
   article697444415.task-FLC.labels: {'emotional_triggering_language': 2, 'topic_shift_misrepresentation': 1, 'doubt_uncertainty_injection': 2}
   article795689901.task-FLC.labels: {'gain_loss_exaggeration': 

### Обработка англоязычного корпуса.

Изначальный [датасет](https://github.com/marcogdepinto/en_corpus) нужно локализовать на русский

#### Перевод через Yandex переводчик

In [15]:
import re, time
from pathlib import Path
import pandas as pd
from tqdm import tqdm
from rapidfuzz import fuzz
import translators as ts

BASE_DIR = Path(MAIN_DIR / "en_corpus/datasets")
ARTICLES_DIR = BASE_DIR / "train-articles"
LABELS_FLC_DIR = BASE_DIR / "train-labels-FLC"

OUTPUT_DIR = Path(MAIN_DIR / "processed_ru")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_CSV = OUTPUT_DIR / "propaganda_ru_unified.csv"
OUTPUT_LOG = OUTPUT_DIR / "processing_log.json"
OUTPUT_STATS = OUTPUT_DIR / "processing_stats.txt"

TRANSLATOR_ENGINE = "yandex"

TECHNIQUE_TO_UNIFIED = {
    "Appeal_to_fear-prejudice": "fear_uncertainty_pressure",
    "Appeal_to_Fear-Prejudice": "fear_uncertainty_pressure",
    "Appeal to fear/prejudice": "fear_uncertainty_pressure",

    "Loaded_Language": "emotional_triggering_language",
    "Doubt": "doubt_uncertainty_injection",

    "Thought-terminating_Cliches": "cognitive_closure_cliches",
    "Thought-terminating cliches": "cognitive_closure_cliches",

    "Bandwagon": "social_proof_pressure",

    "Exaggeration,Minimisation": "gain_loss_exaggeration",
    "Exaggeration, Minimisation": "gain_loss_exaggeration",
    "Exaggeration/Minimisation": "gain_loss_exaggeration",

    "Causal_Oversimplification": "causal_fact_distortion",
    "Causal Oversimplification": "causal_fact_distortion",
    "Black-and-White_Fallacy": "causal_fact_distortion",
    "Black-and-White Fallacy": "causal_fact_distortion",

    "Appeal_to_Authority": "authority_claim_pressure",
    "Appeal to Authority": "authority_claim_pressure",

    "Whataboutism": "topic_shift_misrepresentation",
    "Straw_Men": "topic_shift_misrepresentation",
    "Straw Men": "topic_shift_misrepresentation",
    "Red_Herring": "topic_shift_misrepresentation",
    "Red Herring": "topic_shift_misrepresentation",

    "Repetition": "directive_action_pressure",
}

EXCLUDED_TECHNIQUES = {
    "Name_Calling,Labeling", "Name_Calling", "Labeling",
    "Flag-Waving", "Flag_Waving", "Flag Waving",
    "Slogans",
    "Reductio_ad_hitlerum", "Reductio ad hitlerum",
    "Obfuscation,Intentional_Vagueness,Confusion",
}


def normalize_technique_name(name: str) -> str:
    if not name:
        return ""
    s = name.strip().lower()
    s = re.sub(r"[\s,/_\-]+", "_", s)
    s = re.sub(r"_+", "_", s).strip("_")
    return s

def is_cyrillic_text(s, min_letters=20, min_ratio=0.25):
    s = "" if pd.isna(s) else str(s)
    letters = re.findall(r"[A-Za-zА-Яа-яЁё]", s)
    if len(letters) < min_letters:
        return False
    cyr = re.findall(r"[А-Яа-яЁё]", s)
    return (len(cyr) / len(letters)) >= min_ratio

NORM_EXCLUDED = {normalize_technique_name(x) for x in EXCLUDED_TECHNIQUES}
NORM_MAP = {normalize_technique_name(k): v for k, v in TECHNIQUE_TO_UNIFIED.items()}

def parse_article_id_from_label_file(path: Path):
    m = re.search(r"article(\d+)", path.name)
    return m.group(1) if m else None

def read_article_text(article_id: str):
    p = ARTICLES_DIR / f"article{article_id}.txt"
    if not p.exists():
        return None
    return p.read_text(encoding="utf-8", errors="ignore")

def read_flc_labels(path: Path):
    rows = []
    raw = path.read_text(encoding="utf-8", errors="ignore")
    for line in raw.splitlines():
        line = line.strip()
        if not line:
            continue
        parts = line.split("\t")
        if len(parts) < 4:
            continue
        try:
            art, tech, start, end = parts[0], parts[1], int(parts[2]), int(parts[3])
        except:
            continue
        rows.append({
            "article_id_raw": art,
            "technique": tech,
            "technique_norm": normalize_technique_name(tech),
            "span_start_en": start,
            "span_end_en": end
        })
    return rows

def translate_safe(text: str, retries=10, sleep_sec=15.0):
    text = (text or "").strip()
    if not text:
        return ""

    for i in range(retries):
        try:
            out = ts.translate_text(
                text,
                translator=TRANSLATOR_ENGINE,
                from_language="en",
                to_language="ru"
            )
            out = (out or "").strip()
            if out:
                return out
        except Exception:
            pass
        time.sleep(sleep_sec * (i + 1))

    return text

def find_span(text_ru, span_ru_mt, threshold=88):
    if not text_ru or not span_ru_mt:
        return -1, -1, "low_confidence", ""

    p = text_ru.find(span_ru_mt)
    if p != -1:
        return p, p + len(span_ru_mt), "high_confidence", text_ru[p:p+len(span_ru_mt)]

    t, s = text_ru.lower(), span_ru_mt.lower()
    p = t.find(s)
    if p != -1:
        found = text_ru[p:p+len(span_ru_mt)]
        return p, p + len(found), "medium_confidence", found

    L = len(s)
    if L < 4 or len(t) < L:
        return -1, -1, "low_confidence", ""

    best_score, best_pos = -1, -1
    step = max(1, L // 5)
    for i in range(0, len(t) - L + 1, step):
        chunk = t[i:i+L]
        sc = fuzz.ratio(chunk, s)
        if sc > best_score:
            best_score, best_pos = sc, i

    if best_score >= threshold:
        found = text_ru[best_pos:best_pos+L]
        return best_pos, best_pos + L, "medium_confidence", found

    return -1, -1, "low_confidence", ""

def process_dataset():
    records = []
    article_ru_cache = {}
    span_ru_cache = {}

    label_files = sorted(LABELS_FLC_DIR.glob("*.task-FLC.labels"))
    print(f"Найдено файлов разметки: {len(label_files)}")
    print(f"Translator engine: {TRANSLATOR_ENGINE}")

    for lf in tqdm(label_files, desc=f"Processing FLC ({TRANSLATOR_ENGINE})"):
        article_id = parse_article_id_from_label_file(lf)
        if not article_id:
            continue

        text_en = read_article_text(article_id)
        if text_en is None:
            continue

        rows = read_flc_labels(lf)
        if not rows:
            continue

        if article_id not in article_ru_cache:
            article_ru_cache[article_id] = translate_safe(text_en)
        text_ru = article_ru_cache[article_id]

        not_equal = text_ru.strip() != text_en.strip()
        looks_ru = is_cyrillic_text(text_ru)
        translation_ok = bool(not_equal and looks_ru)

        for r in rows:
            tech_norm = r["technique_norm"]
            if tech_norm in NORM_EXCLUDED:
                continue

            unified = NORM_MAP.get(tech_norm, None)
            if unified is None:
                continue

            s, e = r["span_start_en"], r["span_end_en"]
            if not (0 <= s < e <= len(text_en)):
                continue

            span_en = text_en[s:e]

            if span_en not in span_ru_cache:
                span_ru_cache[span_en] = translate_safe(span_en)
            span_ru_mt = span_ru_cache[span_en]

            span_start_ru, span_end_ru, conf, found_span_ru = find_span(text_ru, span_ru_mt, threshold=88)
            sample_weight = 1.0 if conf == "high_confidence" else (0.75 if conf == "medium_confidence" else 0.5)

            records.append({
                "doc_id": article_id,
                "source_file": lf.name,

                "text_en": text_en,
                "text_ru": text_ru,

                "technique_en": r["technique"],
                "unified_label": unified,

                "span_start_en": s,
                "span_end_en": e,
                "span_text_en": span_en,

                "span_text_ru_mt": span_ru_mt,
                "found_span_ru": found_span_ru,
                "span_start_ru": span_start_ru,
                "span_end_ru": span_end_ru,

                "translation_ok": translation_ok,
                "article_translated": translation_ok,
                "translated": translation_ok,

                "alignment_confidence": conf,
                "sample_weight": sample_weight,
                "binary_label": 1,

                "translator_engine": TRANSLATOR_ENGINE
            })

    df_out = pd.DataFrame(records)
    df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
    return df_out


df_out = process_dataset()
print("Saved:", OUTPUT_CSV, "rows:", len(df_out))
print(df_out.head(3)[[
    "doc_id","translator_engine","translation_ok","article_translated",
    "text_en","text_ru","span_text_en","span_text_ru_mt","found_span_ru","alignment_confidence"
]])

Найдено файлов разметки: 350
Translator engine: yandex


Processing FLC (yandex):   0%|          | 0/350 [00:01<?, ?it/s]


KeyboardInterrupt: 

### Проверка корректности перевода

In [16]:
import pandas as pd

df_out = pd.read_csv(OUTPUT_CSV)

print(f"Сохранено строк: {len(df_out)}")
print(f"Файл: {OUTPUT_CSV}")
print(f"Лог: {OUTPUT_LOG}")
print(f"Статистика: {OUTPUT_STATS}")

if not df_out.empty:
    print(f"\nПервые 3 строки:")
    print(df_out[['technique_en', 'unified_label', 'alignment_confidence']].head(3))
    
    print(f"\nРаспределение меток:")
    for label, count in df_out['unified_label'].value_counts().items():
        print(f"   {label}: {count}")
    
    print(f"\nКачество выравнивания:")
    for conf, count in df_out['alignment_confidence'].value_counts().items():
        print(f"   {conf}: {count}")
else:
    print("\nDataFrame пуст. Проверьте пути к файлам.")

Сохранено строк: 4512
Файл: /home/ksenia/manipulative-language-detector/processed_ru/propaganda_ru_unified.csv
Лог: /home/ksenia/manipulative-language-detector/processed_ru/processing_log.json
Статистика: /home/ksenia/manipulative-language-detector/processed_ru/processing_stats.txt

Первые 3 строки:
              technique_en                  unified_label alignment_confidence
0  Black-and-White_Fallacy         causal_fact_distortion    medium_confidence
1          Loaded_Language  emotional_triggering_language    medium_confidence
2          Loaded_Language  emotional_triggering_language      high_confidence

Распределение меток:
   emotional_triggering_language: 2114
   directive_action_pressure: 571
   doubt_uncertainty_injection: 490
   gain_loss_exaggeration: 478
   causal_fact_distortion: 309
   fear_uncertainty_pressure: 239
   authority_claim_pressure: 116
   topic_shift_misrepresentation: 103
   cognitive_closure_cliches: 79
   social_proof_pressure: 13

Качество выравнивания:

### Проверка перевода

In [20]:
def is_cyrillic_text(s, min_letters=20, min_ratio=0.25):
    s = "" if pd.isna(s) else str(s)
    letters = re.findall(r"[A-Za-zА-Яа-яЁё]", s)
    if len(letters) < min_letters:
        return False
    cyr = re.findall(r"[А-Яа-яЁё]", s)
    return (len(cyr) / len(letters)) >= min_ratio


df = pd.read_csv(OUTPUT_CSV)

not_equal = df["text_ru"].fillna("").str.strip() != df["text_en"].fillna("").str.strip()
looks_ru = df["text_ru"].apply(is_cyrillic_text)
df["translation_ok"] = not_equal & looks_ru
df["article_translated"] = df["translation_ok"]
df["translated"] = df["translation_ok"]

print("Переведено спанов", int(df["translation_ok"].sum()), "/", len(df))

art = df.groupby("doc_id", as_index=False)["translation_ok"].max()
print("Уникальных статей переведено:", int(art["translation_ok"].sum()), "/", len(art))

Переведено спанов 2920 / 4512
Уникальных статей переведено: 292 / 337


In [23]:
total_spans = len(df)

print(f"\nКачество выравнивания:")
alignment_counts = df['alignment_confidence'].value_counts()
for conf in ['high_confidence', 'medium_confidence', 'low_confidence']:
    count = alignment_counts.get(conf, 0)
    pct = count / total_spans * 100
    print(f"   {conf}: {count} ({pct:.1f}%)")


print(f"\nКачество по классам:")
for label in sorted(df['unified_label'].unique()):
    subset = df[df['unified_label'] == label]
    total = len(subset)
    high_med = subset['alignment_confidence'].isin(['high_confidence', 'medium_confidence']).sum()
    pct = high_med / total * 100 if total > 0 else 0
    
    article_translated = subset['article_translated'].sum()
    article_pct = article_translated / total * 100 if total > 0 else 0
    
    print(f"   {label}:")
    print(f"      Всего: {total}, Выровнено хорошо: {high_med} ({pct:.1f}%), Статья переведена: {article_translated} ({article_pct:.1f}%)")


print(f"\nПримеры проблемных спанов (low_confidence + статья не перевелась):")
problematic = df[(df['alignment_confidence'] == 'low_confidence') & 
                 (df['article_translated'] == False)]
print(f"Всего таких спанов: {len(problematic)}")

if len(problematic) > 0:
    print(f"\n   Примеры (первые 5):")
    for _, row in problematic.head(5).iterrows():
        print(f"   [{row['unified_label']}] EN: {row['span_text_en'][:80]}...")
        print(f"   MT: {row['span_text_ru_mt'][:80]}...")


not_translated_ids = df[~df['article_translated']]['doc_id'].unique()
print(f"\nID не переведённых статей:")
print(f"   {list(not_translated_ids[:20])}")


with open(f"{OUTPUT_DIR}/not_translated_articles.txt", 'w') as f:
    for aid in not_translated_ids:
        f.write(f"article{aid}\n")


Качество выравнивания:
   high_confidence: 785 (17.4%)
   medium_confidence: 557 (12.3%)
   low_confidence: 3170 (70.3%)

Качество по классам:
   authority_claim_pressure:
      Всего: 116, Выровнено хорошо: 54 (46.6%), Статья переведена: 88 (75.9%)
   causal_fact_distortion:
      Всего: 309, Выровнено хорошо: 133 (43.0%), Статья переведена: 209 (67.6%)
   cognitive_closure_cliches:
      Всего: 79, Выровнено хорошо: 30 (38.0%), Статья переведена: 56 (70.9%)
   directive_action_pressure:
      Всего: 571, Выровнено хорошо: 241 (42.2%), Статья переведена: 383 (67.1%)
   doubt_uncertainty_injection:
      Всего: 490, Выровнено хорошо: 181 (36.9%), Статья переведена: 314 (64.1%)
   emotional_triggering_language:
      Всего: 2114, Выровнено хорошо: 417 (19.7%), Статья переведена: 1305 (61.7%)
   fear_uncertainty_pressure:
      Всего: 239, Выровнено хорошо: 98 (41.0%), Статья переведена: 164 (68.6%)
   gain_loss_exaggeration:
      Всего: 478, Выровнено хорошо: 132 (27.6%), Статья перев

In [24]:
sample = df.sample(30, random_state=42)[[
    "technique_en", "unified_label", "alignment_confidence",
    "span_text_en", "span_text_ru_mt", "found_span_ru"
]]
sample

,technique_en,unified_label,alignment_confidence,span_text_en,span_text_ru_mt,found_span_ru
2028,Loaded_Language,emotional_triggering_language,low_confidence,cowardice in journalism,трусость в журналистике,NaN
544,Whataboutism,topic_shift_misrepresentation,medium_confidence,"don’t focus on me, focus on the destructive Ra...","не зацикливайтесь на мне, сосредоточьтесь на р...","зацикливайся на мне, сосредоточься на разрушит..."
157,Appeal_to_fear-prejudice,fear_uncertainty_pressure,low_confidence,"A dark, impenetrable and “irreversible” winter...","Наступит темная, непроницаемая и “необратимая”...",NaN
274,Loaded_Language,emotional_triggering_language,low_confidence,grotesque,гротескный,NaN
538,Loaded_Language,emotional_triggering_language,low_confidence,A great injustice has been done.,Произошла великая несправедливость.,NaN
179,Straw_Men,topic_shift_misrepresentation,low_confidence,He mentions the Holy Father's letter to the Bu...,Он упоминает письмо Святого Отца епископам Буэ...,NaN
3616,Appeal_to_Authority,authority_claim_pressure,low_confidence,"Carlson cited Dr. Robert Epstein, a social sci...","Карлсон процитировал доктора Роберта Эпштейна,...",NaN
1670,Appeal_to_fear-prejudice,fear_uncertainty_pressure,high_confidence,we shall fight the Jews and the Americans unti...,мы будем сражаться с евреями и американцами до...,мы будем сражаться с евреями и американцами до...
2917,Appeal_to_Authority,authority_claim_pressure,low_confidence,I’ve looked at all those tweets and they don’t...,"Я просмотрел все эти твиты, и они ни о чем не ...",NaN
2274,Loaded_Language,emotional_triggering_language,low_confidence,grave allegations,серьезные обвинения,NaN


In [25]:
df.describe()

,doc_id,span_start_en,span_end_en,span_start_ru,span_end_ru,sample_weight,binary_label
count,4.512000e+03,4512.000000,4512.000000,4512.00000,4512.000000,4512.000000,4512.0
mean,1.211218e+09,5254.788564,5307.854832,764.61680,785.101950,0.617852,1.0
std,1.704360e+09,6589.634556,6594.198571,1614.56134,1640.898244,0.193209,0.0
min,1.111111e+08,0.000000,5.000000,-1.00000,-1.000000,0.500000,1.0
25%,7.303894e+08,1378.750000,1414.000000,-1.00000,-1.000000,0.500000,1.0
50%,7.669423e+08,3133.500000,3183.000000,-1.00000,-1.000000,0.500000,1.0
75%,7.870023e+08,6380.000000,6428.250000,470.25000,535.500000,0.750000,1.0
max,7.804147e+09,47271.000000,47307.000000,9860.00000,9900.000000,1.000000,1.0


In [26]:
df.head(5)

,doc_id,source_file,text_en,text_ru,technique_en,unified_label,span_start_en,span_end_en,span_text_en,span_text_ru_mt,found_span_ru,span_start_ru,span_end_ru,translation_ok,article_translated,translated,alignment_confidence,sample_weight,binary_label,translator_engine
0,111111112,article111111112.task-FLC.labels,US bloggers banned from entering UK\n\nTwo pro...,Американским блогерам запрещен въезд в Великоб...,Black-and-White_Fallacy,causal_fact_distortion,476,556,We condemn all those whose behaviours and view...,"Мы осуждаем всех тех, чье поведение и взгляды ...","ил: ""Мы осуждаем всех, чье поведение и взгляды...",528,608,True,True,True,medium_confidence,0.75,1,yandex
1,111111112,article111111112.task-FLC.labels,US bloggers banned from entering UK\n\nTwo pro...,Американским блогерам запрещен въезд в Великоб...,Loaded_Language,emotional_triggering_language,958,1015,the nation that gave the world the Magna Carta...,"нация, подарившая миру Великую хартию вольност...","нация, подарившая миру Великую Хартию вольност...",1061,1119,True,True,True,medium_confidence,0.75,1,yandex
2,111111112,article111111112.task-FLC.labels,US bloggers banned from entering UK\n\nTwo pro...,Американским блогерам запрещен въезд в Великоб...,Loaded_Language,emotional_triggering_language,2095,2125,"delighted"" with the decision.\n","в восторге"" от этого решения.","в восторге"" от этого решения.",2306,2335,True,True,True,high_confidence,1.00,1,yandex
3,111111112,article111111112.task-FLC.labels,US bloggers banned from entering UK\n\nTwo pro...,Американским блогерам запрещен въезд в Великоб...,Loaded_Language,emotional_triggering_language,911,942,a striking blow against freedom,сокрушительный удар по свободе,NaN,-1,-1,True,True,True,low_confidence,0.50,1,yandex
4,111111113,article111111113.task-FLC.labels,Kate Steinle's death at the hands of a Mexican...,Смерть Кейт Стейнл от рук гражданина Мексики с...,Loaded_Language,emotional_triggering_language,1396,1430,hair trigger in single-action mode,автоматический спусковой крючок в режиме одино...,NaN,-1,-1,True,True,True,low_confidence,0.50,1,yandex


### 50 случайных примеров из категории low_confidence

In [27]:
low_50 = (
    df[df["alignment_confidence"] == "low_confidence"]
    .sample(n=50, random_state=42)
    .copy()
)

cols = [
    "doc_id",
    "text_en",
    "text_ru",
    "technique_en",
    "span_text_en",
    "span_text_ru_mt",
    "found_span_ru",
    "alignment_confidence",
    "article_translated",
    "translated",
]

low_50[cols]

,doc_id,text_en,text_ru,technique_en,span_text_en,span_text_ru_mt,found_span_ru,alignment_confidence,article_translated,translated
368,703698295,SO MUCH FOR MERCY & DIALOGUE: Member of Vatica...,SO MUCH FOR MERCY & DIALOGUE: Member of Vatica...,"Exaggeration,Minimisation",little incident,небольшой инцидент,NaN,low_confidence,False,False
3156,7804147009,Bishop Morlino Targets ‘Homosexual Subculture’...,Bishop Morlino Targets ‘Homosexual Subculture’...,Loaded_Language,sins and outrages,грехи и бесчинства,NaN,low_confidence,False,False
1465,755459860,Here We Go Again - Leaked Documents: White Hou...,И снова утечка документов: Белый Дом планирует...,Loaded_Language,despise,презирать,NaN,low_confidence,True,True
1397,740356006,Rep. Danny Davis was For/Against/For/Against F...,Член Палаты представителей Дэнни Дэвис был за/...,"Exaggeration,Minimisation",outstanding work”,выдающаяся работа”,NaN,low_confidence,True,True
461,706600938,Did Saint Francis Predict Pope Francis?\n\nTra...,Предсказал ли святой Франциск папу Франциска?\...,Loaded_Language,the malice of the wicked will increase.\nThe d...,злоба нечестивцев будет возрастать.\nДьяволы о...,NaN,low_confidence,True,True
1644,758756657,Islamizing the Schools: The Case of West Virgi...,Исламизация школ: Пример Западной Вирджинии\n\...,Repetition,kill,убивать,NaN,low_confidence,True,True
2491,769962236,Communist Door Boy Attacks Teen Trump Supporte...,"Швейцар-коммунист напал на подростка, поддержи...",Loaded_Language,"tercation,","теркация,",NaN,low_confidence,True,True
4465,999001621,This Guardian Fake News Story Proves That The ...,This Guardian Fake News Story Proves That The ...,Loaded_Language,would explode,взорвался бы,NaN,low_confidence,False,False
2092,764609985,"Clinton Email IG Report Rips FBI, Comey, & Lyn...","Clinton Email IG Report Rips FBI, Comey, & Lyn...",Loaded_Language,"unpersuasive,","неубедительный,",NaN,low_confidence,False,False
295,701553469,Jewish and Pro-Israel Students Kicked Off Univ...,Еврейских и произраильски настроенных студенто...,Loaded_Language,the absurd attacks,абсурдные нападения,NaN,low_confidence,True,True


### Очистка непереведенных данных и модификации

In [29]:
import re
import numpy as np
import pandas as pd
from pathlib import Path
from rapidfuzz import fuzz
from sentence_transformers import SentenceTransformer
import pymorphy2

INPUT_CSV = Path(OUTPUT_DIR / "propaganda_ru_unified.csv")
OUTPUT_CSV = Path(OUTPUT_DIR / "propaganda_ru_unified_clean_aligned.csv")

EMB_MODEL = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
FUZZ_THRESHOLD = 86
SEM_THRESHOLD = 0.62
WINDOW_CHARS = 1400


morph = pymorphy2.MorphAnalyzer()
embedder = SentenceTransformer(EMB_MODEL)


def is_cyrillic_text(s: str, min_letters=20, min_ratio=0.25) -> bool:
    s = "" if pd.isna(s) else str(s)
    letters = re.findall(r"[A-Za-zА-Яа-яЁё]", s)
    if len(letters) < min_letters:
        return False
    cyr = re.findall(r"[А-Яа-яЁё]", s)
    return (len(cyr) / len(letters)) >= min_ratio


def normalize_text(s: str) -> str:
    s = (s or "").lower().strip()
    s = s.replace("ё", "е")
    s = re.sub(r"[\"“”«»'`]", "", s)
    s = re.sub(r"[^a-zа-я0-9\s\-]", " ", s, flags=re.IGNORECASE)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def lemmatize_ru(s: str) -> str:
    toks = re.findall(r"[а-яa-z0-9\-]+", normalize_text(s), flags=re.IGNORECASE)
    lemmas = []
    for t in toks:
        if re.search(r"[а-я]", t, flags=re.IGNORECASE):
            lemmas.append(morph.parse(t)[0].normal_form)
        else:
            lemmas.append(t)
    return " ".join(lemmas)


def char_windows(text: str, target_len: int, step: int = None):
    if target_len <= 0 or len(text) < 3:
        return []
    step = step or max(1, target_len // 4)
    wins = []
    for i in range(0, max(1, len(text) - target_len + 1), step):
        wins.append((i, text[i:i+target_len]))
    if wins and wins[-1][0] + target_len < len(text):
        i = len(text) - target_len
        wins.append((i, text[i:i+target_len]))
    return wins


def stage1_norm_lemma_exact(text_ru: str, span_ru_mt: str):
    p = text_ru.find(span_ru_mt)
    if p != -1:
        return p, p + len(span_ru_mt), "high_confidence", text_ru[p:p+len(span_ru_mt)]

    t_low, s_low = text_ru.lower(), span_ru_mt.lower()
    p = t_low.find(s_low)
    if p != -1:
        found = text_ru[p:p+len(span_ru_mt)]
        return p, p + len(found), "high_confidence", found

    t_norm = normalize_text(text_ru)
    s_norm = normalize_text(span_ru_mt)
    if s_norm and s_norm in t_norm:
        pass

    t_lem = lemmatize_ru(text_ru)
    s_lem = lemmatize_ru(span_ru_mt)
    if s_lem and s_lem in t_lem:
        return -1, -1, "lemma_match_only", ""

    return -1, -1, "no_match", ""


def stage2_context_fuzzy(text_ru: str, span_ru_mt: str, expected_pos: int = None, threshold=FUZZ_THRESHOLD):
    if not text_ru or not span_ru_mt:
        return -1, -1, "low_confidence", ""

    L = len(span_ru_mt)
    if L < 4 or len(text_ru) < L:
        return -1, -1, "low_confidence", ""

    t = text_ru
    offset = 0
    if expected_pos is not None and expected_pos >= 0:
        left = max(0, expected_pos - WINDOW_CHARS // 2)
        right = min(len(text_ru), expected_pos + WINDOW_CHARS // 2)
        if right - left > L:
            t = text_ru[left:right]
            offset = left

    best_sc, best_pos = -1, -1
    step = max(1, L // 6)
    s = span_ru_mt.lower()

    for i in range(0, len(t) - L + 1, step):
        cand = t[i:i+L]
        sc = fuzz.ratio(cand.lower(), s)
        if sc > best_sc:
            best_sc, best_pos = sc, i

    if best_sc >= threshold:
        abs_pos = offset + best_pos
        found = text_ru[abs_pos:abs_pos+L]
        conf = "high_confidence" if best_sc >= 93 else "medium_confidence"
        return abs_pos, abs_pos + L, conf, found

    return -1, -1, "low_confidence", ""


def stage3_semantic_align(text_ru: str, span_ru_mt: str, threshold=SEM_THRESHOLD):
    if not text_ru or not span_ru_mt:
        return -1, -1, "low_confidence", ""

    target_len = max(20, len(span_ru_mt))
    wins = char_windows(text_ru, target_len=target_len, step=max(5, target_len // 5))
    if not wins:
        return -1, -1, "low_confidence", ""

    queries = [span_ru_mt]
    docs = [w[1] for w in wins]

    q_emb = embedder.encode(queries, normalize_embeddings=True)
    d_emb = embedder.encode(docs, normalize_embeddings=True)
    sims = (d_emb @ q_emb.T).reshape(-1)

    j = int(np.argmax(sims))
    best_sim = float(sims[j])

    if best_sim >= threshold:
        start = wins[j][0]
        end = start + len(wins[j][1])
        found = text_ru[start:end]
        conf = "medium_confidence" if best_sim < 0.72 else "high_confidence"
        return start, end, conf, found

    return -1, -1, "low_confidence", ""


def adaptive_align(text_ru: str, span_ru_mt: str, approx_pos: int = None):
    s, e, conf, found = stage1_norm_lemma_exact(text_ru, span_ru_mt)
    if s != -1:
        return s, e, conf, found, "stage1_exact"

    s, e, conf, found = stage2_context_fuzzy(text_ru, span_ru_mt, expected_pos=approx_pos)
    if s != -1:
        return s, e, conf, found, "stage2_fuzzy"

    s, e, conf, found = stage3_semantic_align(text_ru, span_ru_mt)
    if s != -1:
        return s, e, conf, found, "stage3_semantic"

    return -1, -1, "low_confidence", "", "failed"



df = pd.read_csv(INPUT_CSV)
print("Loaded:", len(df))


keep_mask = (
    (df["text_ru"].fillna("").str.strip() != "") &
    (df["translation_ok"].fillna(False).astype(bool))
)
if "translated" in df.columns:
    keep_mask &= df["translated"].fillna(False).astype(bool)


keep_mask &= df["text_ru"].fillna("").apply(is_cyrillic_text)

df = df[keep_mask].copy()
print("After translation filter:", len(df))


used_stage = []
new_conf = []
new_s = []
new_e = []
new_found = []
new_w = []

for _, row in df.iterrows():
    text_ru = str(row.get("text_ru", "") or "")
    span_ru_mt = str(row.get("span_text_ru_mt", "") or "")
    approx = int(row["span_start_ru"]) if pd.notna(row.get("span_start_ru")) else -1

    s, e, conf, found, stage = adaptive_align(text_ru, span_ru_mt, approx_pos=approx)

    new_s.append(s)
    new_e.append(e)
    new_conf.append(conf)
    new_found.append(found)
    used_stage.append(stage)
    new_w.append(1.0 if conf == "high_confidence" else (0.75 if conf == "medium_confidence" else 0.5))

df["span_start_ru"] = new_s
df["span_end_ru"] = new_e
df["alignment_confidence"] = new_conf
df["found_span_ru"] = new_found
df["sample_weight"] = new_w
df["alignment_stage"] = used_stage

df = df[df["alignment_confidence"].isin(["high_confidence", "medium_confidence"])].copy()
print("After alignment filter:", len(df))

df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("Сохранено:", OUTPUT_CSV)

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/9.08M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Loaded: 4512
After translation filter: 2920


KeyboardInterrupt: 

### Проверка перевода и выравнивания спанов после обновленного алгоритма

In [30]:
def is_cyrillic_text(s, min_letters=20, min_ratio=0.25):
    s = "" if pd.isna(s) else str(s)
    letters = re.findall(r"[A-Za-zА-Яа-яЁё]", s)
    if len(letters) < min_letters:
        return False
    cyr = re.findall(r"[А-Яа-яЁё]", s)
    return (len(cyr) / len(letters)) >= min_ratio


df = pd.read_csv(OUTPUT_CSV)

not_equal = df["text_ru"].fillna("").str.strip() != df["text_en"].fillna("").str.strip()
looks_ru = df["text_ru"].apply(is_cyrillic_text)
df["translation_ok"] = not_equal & looks_ru
df["article_translated"] = df["translation_ok"]
df["translated"] = df["translation_ok"]

print("Переведено спанов", int(df["translation_ok"].sum()), "/", len(df))

art = df.groupby("doc_id", as_index=False)["translation_ok"].max()
print("Уникальных статей переведено:", int(art["translation_ok"].sum()), "/", len(art))

Переведено спанов 2887 / 2887
Уникальных статей переведено: 292 / 292


In [31]:
total_spans = len(df)

print(f"\nКачество выравнивания:")
alignment_counts = df['alignment_confidence'].value_counts()
for conf in ['high_confidence', 'medium_confidence', 'low_confidence']:
    count = alignment_counts.get(conf, 0)
    pct = count / total_spans * 100
    print(f"   {conf}: {count} ({pct:.1f}%)")


print(f"\nКачество по классам:")
for label in sorted(df['unified_label'].unique()):
    subset = df[df['unified_label'] == label]
    total = len(subset)
    high_med = subset['alignment_confidence'].isin(['high_confidence', 'medium_confidence']).sum()
    pct = high_med / total * 100 if total > 0 else 0
    
    article_translated = subset['article_translated'].sum()
    article_pct = article_translated / total * 100 if total > 0 else 0
    
    print(f"   {label}:")
    print(f"      Всего: {total}, Выровнено хорошо: {high_med} ({pct:.1f}%), Статья переведена: {article_translated} ({article_pct:.1f}%)")


print(f"\nПримеры проблемных спанов (low_confidence + статья не перевелась):")
problematic = df[(df['alignment_confidence'] == 'low_confidence') & 
                 (df['article_translated'] == False)]
print(f"Всего таких спанов: {len(problematic)}")

if len(problematic) > 0:
    print(f"\n   Примеры (первые 5):")
    for _, row in problematic.head(5).iterrows():
        print(f"   [{row['unified_label']}] EN: {row['span_text_en'][:80]}...")
        print(f"   MT: {row['span_text_ru_mt'][:80]}...")


not_translated_ids = df[~df['article_translated']]['doc_id'].unique()
print(f"\nID не переведённых статей:")
print(f"   {list(not_translated_ids[:20])}")


with open(f"{OUTPUT_DIR}/not_translated_articles.txt", 'w') as f:
    for aid in not_translated_ids:
        f.write(f"article{aid}\n")


Качество выравнивания:
   high_confidence: 2269 (78.6%)
   medium_confidence: 618 (21.4%)
   low_confidence: 0 (0.0%)

Качество по классам:
   authority_claim_pressure:
      Всего: 88, Выровнено хорошо: 88 (100.0%), Статья переведена: 88 (100.0%)
   causal_fact_distortion:
      Всего: 209, Выровнено хорошо: 209 (100.0%), Статья переведена: 209 (100.0%)
   cognitive_closure_cliches:
      Всего: 55, Выровнено хорошо: 55 (100.0%), Статья переведена: 55 (100.0%)
   directive_action_pressure:
      Всего: 383, Выровнено хорошо: 383 (100.0%), Статья переведена: 383 (100.0%)
   doubt_uncertainty_injection:
      Всего: 312, Выровнено хорошо: 312 (100.0%), Статья переведена: 312 (100.0%)
   emotional_triggering_language:
      Всего: 1277, Выровнено хорошо: 1277 (100.0%), Статья переведена: 1277 (100.0%)
   fear_uncertainty_pressure:
      Всего: 164, Выровнено хорошо: 164 (100.0%), Статья переведена: 164 (100.0%)
   gain_loss_exaggeration:
      Всего: 312, Выровнено хорошо: 312 (100.0%),

### Обработка русского корпуса

#### Загрузка русского датасета

In [33]:
import pandas as pd
from datasets import load_dataset

ds_ru = load_dataset("robinhad/manipulation-detection", split="train")
print(ds_ru.features)
pd.DataFrame(ds_ru).head(5)

{'id': Value(dtype='string', id=None), 'content': Value(dtype='string', id=None), 'lang': Value(dtype='string', id=None), 'manipulative': Value(dtype='bool', id=None), 'techniques': [Value(dtype='string', id=None)], 'trigger_words': [[Value(dtype='int64', id=None)]]}


,id,content,lang,manipulative,techniques,trigger_words
0,0bb0c7fa-101b-4583-a5f9-9d503339141c,Новий огляд мапи DeepState від російського вій...,uk,True,"[euphoria, loaded_language]","[[27, 63], [65, 88], [90, 183], [186, 308]]"
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,Недавно 95 квартал жёстко поглумился над русск...,ru,True,"[loaded_language, cherry_picking]","[[0, 40], [123, 137], [180, 251], [253, 274]]"
2,e6a427f1-211f-405f-bd8b-70798458d656,🤩\nТим часом йде евакуація Бєлгородського авто...,uk,True,"[loaded_language, euphoria]","[[55, 100]]"
3,1647a352-4cd3-40f6-bfa1-d87d42e34eea,В Україні найближчим часом мають намір посилит...,uk,False,None,None
4,9c01de00-841f-4b50-9407-104e9ffb03bf,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",ru,True,[loaded_language],"[[114, 144]]"


#### 

In [34]:
import ast
import pandas as pd
from pathlib import Path
from datasets import load_dataset
import numpy as np

MAIN_DIR = (Path.cwd()).parent
OUTPUT_DIR = Path(MAIN_DIR / "processed_ru")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_CSV = Path(MAIN_DIR / "ru_corpus/ru_corpus_processed.csv")

label_map = {
    "loaded_language": "emotional_triggering_language",
    "appeal_to_fear": "fear_uncertainty_pressure",
    "euphoria": "gain_loss_exaggeration",
    "glittering_generalities": "gain_loss_exaggeration",
    "cliche": "cognitive_closure_cliches",
    "bandwagon": "social_proof_pressure",
    "cherry_picking": "causal_fact_distortion",
    "fud": "doubt_uncertainty_injection",
    "whataboutism": "topic_shift_misrepresentation",
    "straw_man": "topic_shift_misrepresentation",

    "fear_uncertainty_pressure": "fear_uncertainty_pressure",
    "emotional_triggering_language": "emotional_triggering_language",
    "doubt_uncertainty_injection": "doubt_uncertainty_injection",
    "cognitive_closure_cliches": "cognitive_closure_cliches",
    "social_proof_pressure": "social_proof_pressure",
    "gain_loss_exaggeration": "gain_loss_exaggeration",
    "causal_fact_distortion": "causal_fact_distortion",
    "authority_claim_pressure": "authority_claim_pressure",
    "topic_shift_misrepresentation": "topic_shift_misrepresentation",
    "directive_action_pressure": "directive_action_pressure",
}

def safe_eval(x):
    if isinstance(x, (list, tuple, np.ndarray)):
        return list(x)
    if not isinstance(x, str):
        return []
    try:
        return ast.literal_eval(x)
    except Exception:
        return []


ds = load_dataset("robinhad/manipulation-detection", split="train")
df = ds.to_pandas()

df = df[df["lang"] == "ru"].copy()
df["techniques"] = df["techniques"].apply(safe_eval)
df["trigger_words"] = df["trigger_words"].apply(safe_eval)

rows = []
for _, row in df.iterrows():
    doc_id = row["id"]
    text = row["content"] or ""
    manipulative = bool(row["manipulative"])
    techs = row["techniques"] if isinstance(row["techniques"], list) else []
    spans = row["trigger_words"] if isinstance(row["trigger_words"], list) else []

    if not spans:
        rows.append({
            "doc_id": doc_id,
            "source_file": "",
            "text_en": "",
            "text_ru": text,
            "technique_en": "",
            "unified_label": "",
            "span_start_en": -1,
            "span_end_en": -1,
            "span_text_en": "",
            "span_text_ru_mt": "",
            "found_span_ru": "",
            "span_start_ru": -1,
            "span_end_ru": -1,
            "translation_ok": True,
            "article_translated": True,
            "translated": True,
            "alignment_confidence": "high_confidence",
            "sample_weight": 1.0,
            "binary_label": int(manipulative),
            "translator_engine": "native_ru",
            "alignment_stage": "native"
        })
        continue

    for i, span in enumerate(spans):
        if span is None or len(span) < 2:
            continue
        s, e = int(span[0]), int(span[1])
        found = text[s:e] if 0 <= s < e <= len(text) else ""

        tech = techs[i] if i < len(techs) else ""
        unified = label_map.get(tech, tech)

        rows.append({
            "doc_id": doc_id,
            "source_file": "",
            "text_en": "",
            "text_ru": text,
            "technique_en": tech,
            "unified_label": unified,
            "span_start_en": -1,
            "span_end_en": -1,
            "span_text_en": "",
            "span_text_ru_mt": found,
            "found_span_ru": found,
            "span_start_ru": s,
            "span_end_ru": e,
            "translation_ok": True,
            "article_translated": True,
            "translated": True,
            "alignment_confidence": "high_confidence",
            "sample_weight": 1.0,
            "binary_label": int(manipulative),
            "translator_engine": "native_ru",
            "alignment_stage": "native"
        })

df_out = pd.DataFrame(rows)
df_out.to_csv(OUTPUT_CSV, index=False, encoding="utf-8")
print("Сохранено:", OUTPUT_CSV, "| shape:", df_out.shape)

Сохранено: /home/ksenia/manipulative-language-detector/ru_corpus/ru_corpus_processed.csv | shape: (4833, 21)


In [37]:
print(df_out["unified_label"].value_counts())

unified_label
                                 2490
emotional_triggering_language     971
causal_fact_distortion            323
gain_loss_exaggeration            237
doubt_uncertainty_injection       228
cognitive_closure_cliches         189
topic_shift_misrepresentation     181
fear_uncertainty_pressure         170
social_proof_pressure              44
Name: count, dtype: int64


In [38]:
df_out.head(10)

,doc_id,source_file,text_en,text_ru,technique_en,unified_label,span_start_en,span_end_en,span_text_en,span_text_ru_mt,...,span_start_ru,span_end_ru,translation_ok,article_translated,translated,alignment_confidence,sample_weight,binary_label,translator_engine,alignment_stage
0,7159f802-6f99-4e9d-97bd-6f565a4a0fae,,,Недавно 95 квартал жёстко поглумился над русск...,loaded_language,emotional_triggering_language,-1,-1,,Недавно 95 квартал жёстко поглумился над,...,0,40,True,True,True,high_confidence,1.0,1,native_ru,native
1,7159f802-6f99-4e9d-97bd-6f565a4a0fae,,,Недавно 95 квартал жёстко поглумился над русск...,cherry_picking,causal_fact_distortion,-1,-1,,дегенеративные,...,123,137,True,True,True,high_confidence,1.0,1,native_ru,native
2,7159f802-6f99-4e9d-97bd-6f565a4a0fae,,,Недавно 95 квартал жёстко поглумился над русск...,,,-1,-1,,Как обычно стать на колени и лизать украинскую...,...,180,251,True,True,True,high_confidence,1.0,1,native_ru,native
3,7159f802-6f99-4e9d-97bd-6f565a4a0fae,,,Недавно 95 квартал жёстко поглумился над русск...,,,-1,-1,,Украинцы же как негры,...,253,274,True,True,True,high_confidence,1.0,1,native_ru,native
4,9c01de00-841f-4b50-9407-104e9ffb03bf,,,"Расчёты 122-мм САУ 2С1 ""Гвоздика"" 132-й бригад...",loaded_language,emotional_triggering_language,-1,-1,,уничтожают украинских нацистов,...,114,144,True,True,True,high_confidence,1.0,1,native_ru,native
5,1417fffa-461c-4bf7-bea5-398e8ea5db81,,,"⚡️\nКитай надеется, что пакетное соглашение о ...",,,-1,-1,,,...,-1,-1,True,True,True,high_confidence,1.0,0,native_ru,native
6,ed5d2195-09b4-4837-82eb-b65244c8a7b2,,,⚡️\nПосле освобождения Соледара откроются благ...,cherry_picking,causal_fact_distortion,-1,-1,,После освобождения Соледара откроются благопри...,...,3,66,True,True,True,high_confidence,1.0,1,native_ru,native
7,ed5d2195-09b4-4837-82eb-b65244c8a7b2,,,⚡️\nПосле освобождения Соледара откроются благ...,cliche,cognitive_closure_cliches,-1,-1,,освобождения Артемовска,...,71,94,True,True,True,high_confidence,1.0,1,native_ru,native
8,7d7504ff-295e-435d-b565-ed62d4eebaa0,,,Российское руководство сделало террор инструме...,loaded_language,emotional_triggering_language,-1,-1,,Российское руководство сделало террор инструме...,...,0,80,True,True,True,high_confidence,1.0,1,native_ru,native
9,7d7504ff-295e-435d-b565-ed62d4eebaa0,,,Российское руководство сделало террор инструме...,cherry_picking,causal_fact_distortion,-1,-1,,массовый психоз,...,1034,1049,True,True,True,high_confidence,1.0,1,native_ru,native


In [39]:
df = pd.read_csv(MAIN_DIR / "ru_corpus/ru_corpus_processed.csv")
print(df.columns.tolist())

['doc_id', 'source_file', 'text_en', 'text_ru', 'technique_en', 'unified_label', 'span_start_en', 'span_end_en', 'span_text_en', 'span_text_ru_mt', 'found_span_ru', 'span_start_ru', 'span_end_ru', 'translation_ok', 'article_translated', 'translated', 'alignment_confidence', 'sample_weight', 'binary_label', 'translator_engine', 'alignment_stage']


In [40]:
total_spans = len(df)

print(f"\nКачество выравнивания (RU):")
alignment_counts = df['alignment_confidence'].value_counts()
for conf in ['high_confidence', 'medium_confidence', 'low_confidence']:
    count = alignment_counts.get(conf, 0)
    pct = count / total_spans * 100 if total_spans > 0 else 0
    print(f"   {conf}: {count} ({pct:.1f}%)")

print(f"\nКачество по классам (RU):")
for label in sorted(df['unified_label'].dropna().unique()):
    subset = df[df['unified_label'] == label]
    total = len(subset)
    high_med = subset['alignment_confidence'].isin(['high_confidence', 'medium_confidence']).sum()
    pct = high_med / total * 100 if total > 0 else 0
    print(f"   {label}:")
    print(f"      Всего: {total}, Выровнено хорошо: {high_med} ({pct:.1f}%)")

print(f"\nПримеры проблемных спанов (low_confidence):")
problematic = df[df['alignment_confidence'] == 'low_confidence']
print(f"Всего таких спанов: {len(problematic)}")

if len(problematic) > 0:
    print(f"\n   Примеры (первые 5):")
    for _, row in problematic.head(5).iterrows():
        print(f"   [{row['unified_label']}] span: {row['found_span_ru'][:80]}...")


Качество выравнивания (RU):
   high_confidence: 4833 (100.0%)
   medium_confidence: 0 (0.0%)
   low_confidence: 0 (0.0%)

Качество по классам (RU):
   causal_fact_distortion:
      Всего: 323, Выровнено хорошо: 323 (100.0%)
   cognitive_closure_cliches:
      Всего: 189, Выровнено хорошо: 189 (100.0%)
   doubt_uncertainty_injection:
      Всего: 228, Выровнено хорошо: 228 (100.0%)
   emotional_triggering_language:
      Всего: 971, Выровнено хорошо: 971 (100.0%)
   fear_uncertainty_pressure:
      Всего: 170, Выровнено хорошо: 170 (100.0%)
   gain_loss_exaggeration:
      Всего: 237, Выровнено хорошо: 237 (100.0%)
   social_proof_pressure:
      Всего: 44, Выровнено хорошо: 44 (100.0%)
   topic_shift_misrepresentation:
      Всего: 181, Выровнено хорошо: 181 (100.0%)

Примеры проблемных спанов (low_confidence):
Всего таких спанов: 0


### Объединение двух датасетов

In [41]:
import pandas as pd
from pathlib import Path

MAIN_DIR = (Path.cwd()).parent
RU_PATH = Path(MAIN_DIR / "ru_corpus/ru_corpus_processed.csv")
TR_PATH = Path(MAIN_DIR / "processed_ru/propaganda_ru_unified_clean_aligned.csv")
OUT_PATH = Path(MAIN_DIR / "processed_ru/merged_ru_plus_translated.csv")

assert RU_PATH.exists(), f"Нет файла: {RU_PATH}"
assert TR_PATH.exists(), f"Нет файла: {TR_PATH}"

ru = pd.read_csv(RU_PATH, keep_default_na=False, na_filter=False)
tr = pd.read_csv(TR_PATH, keep_default_na=False, na_filter=False)

cols = [
    "doc_id","source_file","text_en","text_ru","technique_en","unified_label",
    "span_start_en","span_end_en","span_text_en","span_text_ru_mt",
    "found_span_ru","span_start_ru","span_end_ru",
    "translation_ok","article_translated","translated",
    "alignment_confidence","sample_weight","binary_label",
    "translator_engine","alignment_stage"
]

def normalize(df):
    for c in cols:
        if c not in df.columns:
            df[c] = "" if c in [
                "source_file","text_en","span_text_en","span_text_ru_mt",
                "found_span_ru","technique_en","unified_label",
                "translator_engine","alignment_stage"
            ] else -1
    return df[cols]

ru = normalize(ru)
tr = normalize(tr)

merged = pd.concat([tr, ru], ignore_index=True)
merged.to_csv(OUT_PATH, index=False, encoding="utf-8")

print("Сохранено:", OUT_PATH, "| shape:", merged.shape)
print(merged["unified_label"].value_counts())

Сохранено: /home/ksenia/manipulative-language-detector/processed_ru/merged_ru_plus_translated.csv | shape: (7720, 21)
unified_label
                                 2490
emotional_triggering_language    2248
gain_loss_exaggeration            549
doubt_uncertainty_injection       540
causal_fact_distortion            532
directive_action_pressure         383
fear_uncertainty_pressure         334
topic_shift_misrepresentation     258
cognitive_closure_cliches         244
authority_claim_pressure           88
social_proof_pressure              54
Name: count, dtype: int64


Проверка загрузки merge данных

In [42]:
df = pd.read_csv(MAIN_DIR / "processed_ru/merged_ru_plus_translated.csv")
print("Loaded:", df.shape)

total_spans = len(df)

print(f"\nКачество выравнивания (merged):")
alignment_counts = df['alignment_confidence'].value_counts()
for conf in ['high_confidence', 'medium_confidence', 'low_confidence']:
    count = alignment_counts.get(conf, 0)
    pct = count / total_spans * 100 if total_spans > 0 else 0
    print(f"   {conf}: {count} ({pct:.1f}%)")

print(f"\nКачество по классам (merged):")
for label in sorted(df['unified_label'].dropna().unique()):
    subset = df[df['unified_label'] == label]
    total = len(subset)
    high_med = subset['alignment_confidence'].isin(['high_confidence', 'medium_confidence']).sum()
    pct = high_med / total * 100 if total > 0 else 0
    print(f"   {label}:")
    print(f"      Всего: {total}, Выровнено хорошо: {high_med} ({pct:.1f}%)")

print(f"\nПримеры проблемных спанов (low_confidence):")
problematic = df[df['alignment_confidence'] == 'low_confidence']
print(f"Всего таких спанов: {len(problematic)}")

if len(problematic) > 0:
    print(f"\n   Примеры (первые 5):")
    for _, row in problematic.head(5).iterrows():
        print(f"   [{row['unified_label']}] span: {row['found_span_ru'][:80]}...")

Loaded: (7720, 21)

Качество выравнивания (merged):
   high_confidence: 7102 (92.0%)
   medium_confidence: 618 (8.0%)
   low_confidence: 0 (0.0%)

Качество по классам (merged):
   authority_claim_pressure:
      Всего: 88, Выровнено хорошо: 88 (100.0%)
   causal_fact_distortion:
      Всего: 532, Выровнено хорошо: 532 (100.0%)
   cognitive_closure_cliches:
      Всего: 244, Выровнено хорошо: 244 (100.0%)
   directive_action_pressure:
      Всего: 383, Выровнено хорошо: 383 (100.0%)
   doubt_uncertainty_injection:
      Всего: 540, Выровнено хорошо: 540 (100.0%)
   emotional_triggering_language:
      Всего: 2248, Выровнено хорошо: 2248 (100.0%)
   fear_uncertainty_pressure:
      Всего: 334, Выровнено хорошо: 334 (100.0%)
   gain_loss_exaggeration:
      Всего: 549, Выровнено хорошо: 549 (100.0%)
   social_proof_pressure:
      Всего: 54, Выровнено хорошо: 54 (100.0%)
   topic_shift_misrepresentation:
      Всего: 258, Выровнено хорошо: 258 (100.0%)

Примеры проблемных спанов (low_conf

## Предобработка датасета

ТОЧКА КОНФИГУРАЦИИ МОДЕЛИ И ТОКЕНИЗАТОРА

In [44]:
from transformers import AutoTokenizer

MODEL_NAME = "DeepPavlov/rubert-base-cased"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print(f"Модель: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/642 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/24.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

Модель: DeepPavlov/rubert-base-cased
Vocab size: 119547


### Строим multi-label по unifed_label

In [45]:
import pandas as pd
from collections import defaultdict
import numpy as np

MERGED_PATH = MAIN_DIR / "processed_ru/merged_ru_plus_translated.csv"
df = pd.read_csv(MERGED_PATH)

df = df[df["alignment_confidence"].isin(["high_confidence", "medium_confidence"])].copy()

labels = sorted(df["unified_label"].dropna().unique().tolist())
label2id = {lab: i for i, lab in enumerate(labels)}
id2label = {i: lab for lab, i in label2id.items()}


grouped = defaultdict(list)
for _, row in df.iterrows():
    if row["span_start_ru"] == -1 or row["span_end_ru"] == -1:
        continue
    grouped[row["doc_id"]].append({
        "start": int(row["span_start_ru"]),
        "end": int(row["span_end_ru"]),
        "label": row["unified_label"]
    })


texts = df.drop_duplicates("doc_id")[["doc_id", "text_ru"]].set_index("doc_id")["text_ru"].to_dict()


def build_multilabel_labels(text, spans, tokenizer, label2id):
    enc = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=False,
        add_special_tokens=True
    )
    offsets = enc["offset_mapping"]
    n_classes = len(label2id)

    multi_labels = np.zeros((len(offsets), n_classes), dtype=np.float32)

    for span in spans:
        s, e, lab = span["start"], span["end"], span["label"]
        if lab not in label2id:
            continue
        lab_id = label2id[lab]

        for i, (ts, te) in enumerate(offsets):
            if ts == te == 0:
                continue
            if te <= s or ts >= e:
                continue
            multi_labels[i, lab_id] = 1.0

    return enc, multi_labels


records = []
for doc_id, text in texts.items():
    spans = grouped.get(doc_id, [])
    enc, multi = build_multilabel_labels(text, spans, tokenizer, label2id)
    records.append({
        "doc_id": doc_id,
        "text": text,
        "input_ids": enc["input_ids"],
        "attention_mask": enc["attention_mask"],
        "multi_labels": multi
    })

multi_df = pd.DataFrame(records)
print("multi-dataset:", multi_df.shape)
print(multi_df.head())

labels = sorted(df["unified_label"].dropna().unique().tolist())
print(f"Уникальных лейблов: {len(labels)}")
print(labels)

multi-dataset: (1967, 5)
      doc_id                                               text  \
0  111111112  Американским блогерам запрещен въезд в Великоб...   
1  111111113  Смерть Кейт Стейнл от рук гражданина Мексики с...   
2  111111122  Выдвижение Кавано на пост президента США стрем...   
3  111111124  Агентства изо всех сил пытаются подготовить ма...   
4  111111131  Северокорейская авантюра Трампа заканчивается ...   

                                           input_ids  \
0  [101, 91369, 94324, 866, 52203, 30396, 845, 33...   
1  [101, 33627, 28701, 72817, 864, 1641, 11451, 2...   
2  [101, 51070, 19336, 22408, 5709, 1469, 3787, 8...   
3  [101, 50345, 52663, 6399, 5563, 23996, 37656, ...   
4  [101, 81239, 21159, 3296, 87639, 626, 37844, 2...   

                                      attention_mask  \
0  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
1  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
2  [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...   
3  [1, 1, 1

Создание label2id + чанкирование

In [46]:
MAX_LEN = 512
STRIDE = 128

def filter_valid_spans(spans, debug=False):
    valid_spans = []
    skipped = 0
    for span in spans:
        lab = span.get("label")
        if not lab or pd.isna(lab) or str(lab).lower() == "nan":
            skipped += 1
            continue
        valid_spans.append(span)
    if debug and skipped > 0:
        print(f"Валидных спанов: {len(valid_spans)}, пропущено: {skipped}")
    return valid_spans

def build_multilabel_chunk(offsets, spans, label2id):
    n_classes = len(label2id)
    chunk_labels = np.zeros((len(offsets), n_classes), dtype=np.float32)

    for span in spans:
        s, e, lab = span["start"], span["end"], span["label"]
        if lab not in label2id:
            continue
        lab_id = label2id[lab]

        for i, (ts, te) in enumerate(offsets):
            if ts == te == 0:
                continue
            if te <= s or ts >= e:
                continue
            chunk_labels[i, lab_id] = 1.0

    for i, (ts, te) in enumerate(offsets):
        if ts == te == 0:
            chunk_labels[i, :] = -100.0

    return chunk_labels

def chunk_record_multilabel(row):
    text = row["text"]
    spans = filter_valid_spans(grouped.get(row["doc_id"], []), debug=False)

    enc_chunks = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LEN,
        stride=STRIDE,
        return_overflowing_tokens=True,
        add_special_tokens=True
    )

    chunks = []
    for i in range(len(enc_chunks["input_ids"])):
        input_ids = enc_chunks["input_ids"][i]
        attention_mask = enc_chunks["attention_mask"][i]
        offsets = enc_chunks["offset_mapping"][i]

        multi_labels = build_multilabel_chunk(offsets, spans, label2id)

        chunks.append({
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": multi_labels,
            "doc_id": row["doc_id"]
        })

    return chunks

chunked = []
for _, row in multi_df.iterrows():
    chunked.extend(chunk_record_multilabel(row))

chunked_df = pd.DataFrame(chunked)
print("Размерность:", chunked_df.shape)
chunked_df.head(10)

Размерность: (2442, 4)


,input_ids,attention_mask,labels,doc_id
0,"[101, 91369, 94324, 866, 52203, 30396, 845, 33...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111112
1,"[101, 35488, 845, 6377, 10758, 166, 6654, 1292...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111112
2,"[101, 33627, 28701, 72817, 864, 1641, 11451, 2...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111113
3,"[101, 4105, 2739, 42488, 27959, 1444, 1469, 80...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111113
4,"[101, 19107, 28699, 1436, 845, 13170, 22108, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111113
5,"[101, 1699, 3370, 33834, 845, 18114, 132, 7944...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111113
6,"[101, 51070, 19336, 22408, 5709, 1469, 3787, 8...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111122
7,"[101, 12462, 2886, 2067, 35165, 6675, 25909, 6...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111122
8,"[101, 8980, 17499, 82062, 9643, 128, 4620, 898...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111122
9,"[101, 793, 6974, 128, 877, 16758, 51316, 6043,...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ...","[[-100.0, -100.0, -100.0, -100.0, -100.0, -100...",111111122


In [47]:
def show_multilabel_examples(df, tokenizer, id2label, max_examples=3):
    shown = 0
    for _, row in df.iterrows():
        labels = row["labels"]

        multi_tokens = np.where((labels > 0.5).sum(axis=1) > 1)[0]
        if len(multi_tokens) == 0:
            continue

        tokens = tokenizer.convert_ids_to_tokens(row["input_ids"])
        print(f"\nDoc_id: {row['doc_id']}")
        for t_idx in multi_tokens[:5]:
            active = np.where(labels[t_idx] > 0.5)[0]
            labs = [id2label[i] for i in active]
            print(f"{tokens[t_idx]:20s} -> {labs}")

        shown += 1
        if shown >= max_examples:
            break


show_multilabel_examples(chunked_df, tokenizer, id2label, max_examples=5)


Doc_id: 111111113
инопланетян          -> ['causal_fact_distortion', 'emotional_triggering_language']
##ина                -> ['causal_fact_distortion', 'emotional_triggering_language']

Doc_id: 111111131
историческим         -> ['directive_action_pressure', 'emotional_triggering_language']

Doc_id: 111111131
историческим         -> ['directive_action_pressure', 'emotional_triggering_language']

Doc_id: 694356862
безрассуд            -> ['causal_fact_distortion', 'emotional_triggering_language']
##ное                -> ['causal_fact_distortion', 'emotional_triggering_language']
умиротвор            -> ['causal_fact_distortion', 'emotional_triggering_language']
##ение               -> ['causal_fact_distortion', 'emotional_triggering_language']

Doc_id: 696264594
Либо                 -> ['causal_fact_distortion', 'fear_uncertainty_pressure']
вы                   -> ['causal_fact_distortion', 'fear_uncertainty_pressure']
поддерживает         -> ['causal_fact_distortion', 'fear_uncertaint

In [ ]:
def show_labeled_tokens_inline(chunked_df, tokenizer, id2label, doc_id, max_chunks=1):
    rows = chunked_df[chunked_df["doc_id"] == doc_id]
    if rows.empty:
        print("Doc_id не найден")
        return

    shown = 0
    for _, row in rows.iterrows():
        input_ids = row["input_ids"]
        labels = row["labels"]
        tokens = tokenizer.convert_ids_to_tokens(input_ids)

        print(f"\n--- doc_id={doc_id} ---")
        for tok, lab_vec in zip(tokens, labels):
            if lab_vec[0] == -100:
                continue
            active = np.where(lab_vec > 0.5)[0]
            if len(active) == 0:
                print(f"{tok}")
            else:
                labs = [id2label[i] for i in active]
                print(f"{tok} -> {labs}")

        shown += 1
        if shown >= max_chunks:
            break


show_labeled_tokens_inline(chunked_df, tokenizer, id2label, doc_id="111111113", max_chunks=1)


--- doc_id=111111113 ---
Смерть
Кейт
Стейн
##л
от
рук
гражданина
Мексики
стала
горячей
точкой
в
дебатах
об
иммиграции
—
вот
история
,
стоящая
за
ее
убийством
Неожид
##анный
оправдательный
приговор
Хосе
Ин
##есу
Гарсии
Сарат
##е
по
делу
о
расстреле
жительницы
Сан
-
Франциско
Кейт
Стейн
##л
вызвал -> ['emotional_triggering_language']
бурю -> ['emotional_triggering_language']
негод -> ['emotional_triggering_language']
##ования -> ['emotional_triggering_language']
в -> ['emotional_triggering_language']
четверг -> ['emotional_triggering_language']
вечером
,
поскольку
ведущие
консерваторы
и
критики
так
называемых
"
городов
-
убежищ -> ['causal_fact_distortion']
" -> ['causal_fact_distortion']
возложили -> ['causal_fact_distortion']
вину -> ['causal_fact_distortion']
за -> ['causal_fact_distortion']
смерть -> ['causal_fact_distortion']
Стейн -> ['causal_fact_distortion']
##л -> ['causal_fact_distortion']
на -> ['causal_fact_distortion']
нелегальную -> ['causal_fact_distortion']
иммигра -> ['

Создание HF Dataset

In [56]:
from datasets import DatasetDict, Dataset, Features, Sequence, Value
import numpy as np


def to_list(x):
    return x.tolist() if hasattr(x, "tolist") else x

chunked_df["labels"] = chunked_df["labels"].apply(to_list)

features = Features({
    "input_ids": Sequence(Value("int64")),
    "attention_mask": Sequence(Value("int64")),
    "labels": Sequence(Sequence(Value("float32"))),
    "doc_id": Value("string"),
})

dataset = Dataset.from_pandas(chunked_df, features=features)


dataset = dataset.remove_columns([c for c in dataset.column_names if c.startswith("__")])


split = dataset.train_test_split(test_size=0.2, seed=42)
train_val = split["train"].train_test_split(test_size=0.15, seed=42)

ds = DatasetDict({
    "train": train_val["train"],      # 68%
    "validation": train_val["test"],  # 12%
    "test": split["test"]             # 20%
})


ds.set_format(
    type="torch",
    columns=["input_ids", "attention_mask", "labels"]
)

print(ds)

OUT_DIR = MAIN_DIR / "ru_token_cls_dataset"
ds.save_to_disk(OUT_DIR)
print(f"Сохранено в {OUT_DIR}")

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'doc_id'],
        num_rows: 1660
    })
    validation: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'doc_id'],
        num_rows: 293
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask', 'labels', 'doc_id'],
        num_rows: 489
    })
})


Saving the dataset (0/1 shards):   0%|          | 0/1660 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/293 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/489 [00:00<?, ? examples/s]

Сохранено в /home/ksenia/manipulative-language-detector/ru_token_cls_dataset


Проверка валидности и размеров

In [54]:
ds.set_format(type=None)

n_classes = len(label2id)

def check_multilabel(ds, n_classes):
    ok = True
    for split in ds:
        for ex in ds[split]:
            labs = np.array(ex["labels"])
            if labs.ndim != 2 or labs.shape[1] != n_classes:
                print(f"[{split}] bad shape: {labs.shape}")
                ok = False
            if not np.isin(labs[labs != -100], [0, 1]).all():
                print(f"[{split}] non-binary values found")
                ok = False
    print("OK" if ok else "Есть ошибки")

check_multilabel(ds, n_classes)

OK


Распределение меток по токенам

In [ ]:
from collections import Counter
import numpy as np

def multilabel_distribution(ds, id2label):
    counts = Counter()
    for split in ds:
        for ex in ds[split]:
            labs = np.array(ex["labels"])
            mask = ~(labs == -100).all(axis=1)
            labs = labs[mask]
            counts.update({
                id2label[i]: int(labs[:, i].sum())
                for i in range(labs.shape[1])
            })
    return counts

counts = multilabel_distribution(ds, id2label)

print("Распределение меток:")
for k, v in counts.most_common():
    print(f"{k}: {v}")

Распределение меток:
emotional_triggering_language: 25157
doubt_uncertainty_injection: 12310
causal_fact_distortion: 11314
gain_loss_exaggeration: 7186
fear_uncertainty_pressure: 6765
topic_shift_misrepresentation: 5282
authority_claim_pressure: 3469
cognitive_closure_cliches: 2714
directive_action_pressure: 1492
social_proof_pressure: 872
